In [ ]:
!pip install flask flask-cors pandas scikit-learn joblib spacy
!python -m spacy download en_core_web_sm

from flask import Flask, request, jsonify
from flask_cors import CORS
import pandas as pd
import spacy
import joblib
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import os

nlp = spacy.load("en_core_web_sm")

def extract_symptoms(text, known_symptoms):
    text = text.lower().strip()
    found_symptoms = []

    for symptom in known_symptoms:
        symptom_clean = symptom.lower().strip().replace('_', ' ')
        if symptom_clean in text:
            found_symptoms.append(symptom)

    return found_symptoms

df = pd.read_csv('dataset.csv')
df.columns = df.columns.str.strip()

# Get all unique symptoms
symptom_cols = [col for col in df.columns if col != 'Disease']
all_symptoms = set()
for col in symptom_cols:
    all_symptoms.update(df[col].dropna().str.strip().unique())

all_symptoms = sorted(list(all_symptoms))
print(f"Total unique symptoms: {len(all_symptoms)}")

# Create binary matrix
binary_data = []
for _, row in df.iterrows():
    row_symptoms = set()
    for col in symptom_cols:
        if pd.notna(row[col]):
            row_symptoms.add(row[col].strip())
    binary_row = {s: 1 if s in row_symptoms else 0 for s in all_symptoms}
    binary_data.append(binary_row)

X = pd.DataFrame(binary_data)
y = df['Disease'].str.strip()

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

# Train
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Accuracy
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

# Save
os.makedirs('models', exist_ok=True)
joblib.dump(model, 'models/disease_model.pkl')
joblib.dump(le, 'models/label_encoder.pkl')
joblib.dump(all_symptoms, 'models/symptom_columns.pkl')

print("Model saved successfully!")
print(f"Sample symptoms: {all_symptoms[:10]}")

model = joblib.load('models/disease_model.pkl')
le = joblib.load('models/label_encoder.pkl')
symptom_columns = joblib.load('models/symptom_columns.pkl')

def predict_disease(symptoms_list):
    # Create input vector
    input_vector = pd.DataFrame([{
        col: 1 if col in symptoms_list else 0
        for col in symptom_columns
    }])

    # Predict
    prediction = model.predict(input_vector)
    probabilities = model.predict_proba(input_vector)[0]

    # Get top 3 diseases
    top3_indices = np.argsort(probabilities)[::-1][:3]
    top3_diseases = [
        {
            "disease": le.inverse_transform([i])[0],
            "confidence": round(float(probabilities[i]) * 100, 2)
        }
        for i in top3_indices
    ]

    return top3_diseases

def get_urgency(disease, severity_df):
    disease_lower = disease.lower()

    # High urgency diseases
    high_urgency = ['heart attack', 'stroke', 'diabetes', 'hypertension',
                    'pneumonia', 'hepatitis', 'tuberculosis', 'malaria',
                    'dengue', 'typhoid', 'jaundice']

    # Medium urgency diseases
    medium_urgency = ['fungal infection', 'allergy', 'gastroenteritis',
                      'bronchial asthma', 'urinary tract infection',
                      'migraine', 'cervical spondylosis', 'paralysis']

    if any(d in disease_lower for d in high_urgency):
        return {
            "level": "red",
            "message": "Seek immediate medical attention!",
            "color": "#DC3545"
        }
    elif any(d in disease_lower for d in medium_urgency):
        return {
            "level": "yellow",
            "message": "Monitor symptoms, consult doctor soon.",
            "color": "#FFC107"
        }
    else:
        return {
            "level": "green",
            "message": "Rest at home and monitor symptoms.",
            "color": "#28A745"
        }

app = Flask(__name__)
CORS(app, resources={r"/*": {"origins": "*"}})

# Load severity data
try:
    severity_df = pd.read_csv('Symptom-severity.csv')
    severity_df.columns = severity_df.columns.str.strip()
    print("Severity data loaded:", severity_df.columns.tolist())
except Exception as e:
    print(f"Error loading severity data: {e}")
    severity_df = pd.DataFrame()

@app.route('/analyze', methods=['POST', 'OPTIONS'])
def analyze():
    if request.method == 'OPTIONS':
        return jsonify({}), 200

    try:
        data = request.get_json()
        user_text = data.get('text', '')
        body_parts = data.get('body_parts', [])

        symptoms = extract_symptoms(user_text, symptom_columns)
        symptoms = list(set(symptoms + body_parts))

        print(f"Symptoms found: {symptoms}")

        if not symptoms:
            return jsonify({
                "error": "No symptoms found. Please describe your symptoms in more detail."
            }), 400

        predictions = predict_disease(symptoms)
        top_disease = predictions[0]['disease']

        print(f"Top disease: {top_disease}")

        urgency = get_urgency(top_disease, severity_df)

        return jsonify({
            "symptoms_found": symptoms,
            "predictions": predictions,
            "urgency": urgency
        })

    except Exception as e:
        print(f"Error in analyze: {e}")
        return jsonify({"error": str(e)}), 500

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "running"})

if __name__ == '__main__':
    app.run(debug=True, port=5000)

  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
Using cached flask_cors-6.0.2-py3-none-any.whl (13 kB)
